In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS aulas_ao_vivo.live_20260309;
CREATE VOLUME IF NOT EXISTS aulas_ao_vivo.live_20260309.files;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import time

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("Live2_Armazenamento_Camadas") \
        .config("spark.sql.shuffle.partitions", "8") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

N_LINHAS = 50_000_000
SEED = 66
START_TS = "2021-01-01 00:00:00"
TAXA_DUPLICIDADE = 0.08 # sempre float, separado por ponto e < 1
PROCESS_DATE = "2026-03-01"

VOLUME_PATH = "/Volumes/aulas_ao_vivo/live_20260309/files"
BASE_PATH = f"{VOLUME_PATH}/live_20260309"
BRONZE_BASE_PATH = f"{BASE_PATH}/bronze"
CSV_PATH = f"{BRONZE_BASE_PATH}/ecommerce/pedidos_csv/dt={PROCESS_DATE}"
JSON_PATH = f"{BRONZE_BASE_PATH}/ecommerce/pedidos_json/dt={PROCESS_DATE}"
PARQUET_PATH = f"{BRONZE_BASE_PATH}/ecommerce/pedidos_parquet/dt={PROCESS_DATE}"

CATALOG_SCHEMA = "aulas_ao_vivo.live_20260309"
TBL_SILVER_PEDIDOS = f"{CATALOG_SCHEMA}.silver_vendas_pedidos"
TBL_SILVER_ITENS = f"{CATALOG_SCHEMA}.silver_vendas_itens_pedido"
TBL_SILVER_PRODUTOS = f"{CATALOG_SCHEMA}.silver_catalogo_produtos"
TBL_GOLD_RECEITA = f"{CATALOG_SCHEMA}.gold_vendas_receita_mensal_produto_uf"
TBL_GOLD_KPIS = f"{CATALOG_SCHEMA}.gold_vendas_kpis_mensais"

def d(df, n=20):
    """Helper de visualização: usa display() no Databricks; senão, df.show()."""
    try:
        display(df)
    except NameError:
        df.show(n, truncate=False)

def show_title(txt):
    print("\n" + "=" * 100)
    print(txt)
    print("=" * 100)

def standardize_column_names(df):
    """
    Padroniza nomes de colunas para snake_case.
    Ex.: "Valor Total (R$)" -> "valor_total_r".
    """
    import re
    cols = df.columns
    new_cols = []
    for c in cols:
        c2 = c.strip().lower()
        c2 = re.sub(r"[^a-z0-9]+", "_", c2)
        c2 = re.sub(r"_+", "_", c2).strip("_")
        new_cols.append(c2)
    out = df
    for old, new in zip(cols, new_cols):
        if old != new:
            out = out.withColumnRenamed(old, new)
    return out

def rm(path):
    try:
        dbutils.fs.rm(path, True)
    except Exception:
        pass

def ls_recursive(path):
    out = []
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return out
    for item in items:
        if item.path.endswith("/"):
            out.extend(ls_recursive(item.path))
        else:
            out.append(item)
    return out

def path_stats(path):
    files = ls_recursive(path)
    total_bytes = sum(x.size for x in files)
    total_mb = round(total_bytes / (1024 * 1024), 4)
    return len(files), total_bytes, total_mb

def benchmark_read(format_name, path, options=None, schema=None):
    reader = spark.read.format(format_name) # instancia um Spark reader (vide doc)
    if options:
        for k, v in options.items():
            reader = reader.option(k, v)
    if schema is not None and format_name in ("csv", "json"):
        reader = reader.schema(schema)
    t0 = time.perf_counter()
    total = reader.load(path).count()
    elapsed = round(time.perf_counter() - t0, 4)
    n_files, total_bytes, total_mb = path_stats(path)
    return (format_name, path, total, n_files, total_bytes, total_mb, elapsed)


In [0]:
from random import randint

def generate_vendas_bronze(
    n_linhas: int,
    seed: int = 42,
    start_ts: str = "2026-03-01 00:00:00",
    taxa_duplicidade: float = 0.0
):
    """
    Gera um DataFrame propositalmente "sujo" (bronze) para demonstrar limpeza mínima.
    Controle de volume: n_linhas.

    EVOLUÇÃO DA LIVE 1:
      - mantemos a mesma função-base da aula anterior
      - adicionamos taxa_duplicidade para injetar duplicidades por sample + union
      - isso sustenta a transição didática para Silver (deduplicação explícita)

    Colunas (intencionalmente inconsistentes):
      - "Id Pedido" (string com espaços)
      - "Data Pedido" (string em formatos mistos)
      - "UF " (string com caixa/espaços inconsistentes)
      - "Produto" (string com caixa/espaços inconsistentes)
      - "Qtd" (string numérica com zeros/espaços)
      - "Valor Total (R$)" (string com separador decimal misto)
      - "Status" (string com variações)
    """
    if n_linhas < 1:
        return spark.createDataFrame([], schema=T.StructType([]))

    if taxa_duplicidade < 0 or taxa_duplicidade >= 1:
        raise ValueError("taxa_duplicidade deve estar entre 0 e 1 (exclusivo do 1).")

    ufs = [
        "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA",
        "PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"
    ]
    produtos = ["Camiseta", "Tenis", "Caderno", "Mochila", "Fone", "Mouse", "Garrafa", "Jaqueta"]

    base = spark.range(n_linhas).withColumn("rnd", F.rand(seed))

    id_raw = F.concat(F.lit("PED"), F.lpad(F.col("id").cast("string"), 8, "0"))
    id_sujo = F.when(F.col("id") % 5 == 0, F.concat(F.lit(" "), id_raw, F.lit(" "))).otherwise(id_raw)

    start_unix = F.unix_timestamp(F.lit(start_ts), "yyyy-MM-dd HH:mm:ss")
    ts = F.to_timestamp(F.from_unixtime(start_unix + (F.col("id") * F.lit(3600))))
    fmt1 = F.date_format(ts, "yyyy-MM-dd HH:mm:ss")
    fmt2 = F.date_format(ts, "dd/MM/yyyy HH:mm")
    fmt3 = F.date_format(ts, "dd/MM/yyyy HH:mm:ss")
    data_suja = (
        F.when(F.col("id") % 3 == 0, fmt1)
         .when(F.col("id") % 3 == 1, fmt2)
         .otherwise(fmt3)
    )

    idx_uf = ((F.col("id") % F.lit(len(ufs))) + F.lit(1)).cast("int")
    idx_prod = ((F.col("id") % F.lit(len(produtos))) + F.lit(1)).cast("int")

    uf_raw = F.element_at(F.array(*[F.lit(x) for x in ufs]), idx_uf)
    uf_sujo = (
        F.when(F.col("id") % 4 == 0, F.lower(uf_raw))
         .when(F.col("id") % 4 == 1, F.concat(F.lit(" "), uf_raw, F.lit(" ")))
         .otherwise(uf_raw)
    )

    prod_raw = F.element_at(F.array(*[F.lit(x) for x in produtos]), idx_prod)
    prod_sujo = (
        F.when(F.col("id") % 4 == 0, F.upper(prod_raw))
         .when(F.col("id") % 4 == 1, F.concat(F.lit(" "), prod_raw, F.lit(" ")))
         .when(F.col("id") % 7 == 0, F.lit(" "))
         .otherwise(prod_raw)
    )

    qty_num = (F.col("id") % F.lit(5)) + F.lit(1)
    qty_str = (
        F.when(F.col("id") % 4 == 0, F.lpad(qty_num.cast("string"), 2, "0"))
         .when(F.col("id") % 9 == 0, F.lit(" "))
         .otherwise(qty_num.cast("string"))
    )

    preco_base = ((F.col("id") % F.lit(20)) + F.lit(1)) * F.lit(10.5)
    valor_num = F.round(preco_base * qty_num.cast("double"), 2)
    valor_dot = F.format_string("%.2f", valor_num)
    valor_comma = F.regexp_replace(valor_dot, "\\.", ",")
    valor_sujo = (
        F.when(F.col("id") % 3 == 0, valor_dot)
         .when(F.col("id") % 3 == 1, valor_comma)
         .otherwise(F.concat(F.lit(" "), valor_comma, F.lit(" ")))
    )

    status_sujo = (
        F.when(F.col("id") % 3 == 0, F.lit("Pago"))
         .when(F.col("id") % 3 == 1, F.lit("CANCELADO"))
         .otherwise(F.lit(" Pendente "))
    )

    df = base.select(
        id_sujo.alias("Id Pedido"),
        data_suja.alias("Data Pedido"),
        uf_sujo.alias("UF "),
        prod_sujo.alias("Produto"),
        qty_str.alias("Qtd"),
        valor_sujo.alias("Valor Total (R$)"),
        status_sujo.alias("Status")
    )

    if taxa_duplicidade > 0:
        duplicadas = df.sample(
            withReplacement=False, 
            fraction=taxa_duplicidade, 
            seed=seed + 999
        )
        df = df.unionByName(duplicadas)

    return df

def generate_produtos_bronze():
    """
    Export sintético do ERP de produtos.
    Mantemos pequenas inconsistências de caixa/espaço para padronizar na Silver.
    """
    rows = [
        (" camiseta ", "Moda", "ativo"),
        ("TENIS", "Moda", "ativo"),
        ("Caderno", "Papelaria", "ativo"),
        (" mochila ", "Acessorios", "ativo"),
        ("Fone", "Eletronicos", "ativo"),
        ("Mouse", "Eletronicos", "ativo"),
        ("garrafa", "Casa", "ativo"),
        ("Jaqueta", "Moda", "ativo")
    ]
    schema = T.StructType([
        T.StructField("Produto", T.StringType(), False),
        T.StructField("Categoria", T.StringType(), False),
        T.StructField("Status Produto", T.StringType(), False)
    ])
    return spark.createDataFrame(rows, schema)


In [0]:
rm(BASE_PATH)

vendas_bronze = generate_vendas_bronze(
    n_linhas=N_LINHAS,
    seed=SEED,
    start_ts=START_TS,
    taxa_duplicidade=TAXA_DUPLICIDADE
)
produtos_bronze = generate_produtos_bronze()

show_title("1) Continuidade da Live anterior — mesmo projeto, bronze evoluído com duplicidades")
print("Linhas BRONZE (após sample + union):", vendas_bronze.count())
print("Colunas BRONZE:", vendas_bronze.columns)
print("Schema BRONZE:")
vendas_bronze.printSchema()

print("Amostra BRONZE:")
d(vendas_bronze.limit(20))

# Guarde esse algoritmo, porque ele é muito útil.
duplicatas_bronze = (
    vendas_bronze
    .select(F.trim(F.col("Id Pedido")).alias("id_pedido"))
    .groupBy("id_pedido")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"), F.asc("id_pedido"))
)

print("Pedidos duplicados no BRONZE:")
d(duplicatas_bronze)

print("Amostra do export de produtos (BRONZE):")
d(produtos_bronze)


In [0]:
"""
PASSO: Salvar o mesmo Bronze em 3 formatos de arquivo.
O QUE OBSERVAR:
  - CSV e JSON cumprem bem o papel de transporte/evidência.
  - Parquet costuma ser mais eficiente para leitura analítica.
VALIDAR:
  - tamanho físico em bytes/MB
  - tempo de leitura simples com count()
SINAL DE ERRO:
  - tirar conclusão absoluta de performance sem contexto; aqui o benchmark é didático.
"""

bronze_write_df = vendas_bronze.repartition(8)

bronze_write_df.write.mode("overwrite").option("header", "true").csv(CSV_PATH)
bronze_write_df.write.mode("overwrite").json(JSON_PATH)
bronze_write_df.write.mode("overwrite").parquet(PARQUET_PATH)

bronze_schema = vendas_bronze.schema

benchmark_rows = [
    benchmark_read("csv", CSV_PATH, options={"header": "true"}, schema=bronze_schema),
    benchmark_read("json", JSON_PATH, schema=bronze_schema),
    benchmark_read("parquet", PARQUET_PATH)
]

benchmark_schema = T.StructType([
    T.StructField("formato", T.StringType(), False),
    T.StructField("path", T.StringType(), False),
    T.StructField("qtd_linhas", T.LongType(), False),
    T.StructField("qtd_arquivos", T.IntegerType(), False),
    T.StructField("tamanho_bytes", T.LongType(), False),
    T.StructField("tamanho_mb", T.DoubleType(), False),
    T.StructField("tempo_leitura_seg", T.DoubleType(), False)
])

bronze_benchmark = (
    spark.createDataFrame(benchmark_rows, benchmark_schema)
    .orderBy(F.asc("tempo_leitura_seg"), F.asc("tamanho_mb"))
)

show_title("2) Comparação de formatos no Bronze")
d(bronze_benchmark)

bronze_benchmark.createOrReplaceTempView("bronze_benchmark")


In [0]:
%sql
SELECT formato, tamanho_mb, tempo_leitura_seg
FROM bronze_benchmark
/*
5_000
formato	  tamanho_mb	tempo_leitura_seg
parquet	  0.0889	    1.4069
json	    0.787	      1.5219
csv	      0.3072	    1.9811
*/

/*
50_000_000
formato	  tamanho_mb	tempo_leitura_seg
parquet	  536.8981	  1.3396
csv	      3059.3463	  7.2906
json	    7861.0753	  45.3859
*/

In [0]:
"""
PASSO: Explicar a organização por camada e por tipo de promessa.
O QUE OBSERVAR:
  - Bronze: arquivo de entrada/evidência
  - Silver: tabela técnica reutilizável
  - Gold: tabela com contrato de consumo
VALIDAR:
  - paths previsíveis
  - nomenclatura estável
SINAL DE ERRO:
  - usar um único lugar para tudo e depender de memória oral.
"""

organizacao = spark.createDataFrame([
    ("Bronze", "arquivo", f"bronze/ecommerce/pedidos_<formato>/dt={PROCESS_DATE}/", "evidência do que chegou"),
    ("Bronze", "arquivo", f"bronze/erp/produtos/dt={PROCESS_DATE}/", "export recebido da fonte"),
    ("Silver", "tabela", "silver_vendas_pedidos", "pedido padronizado e deduplicado"),
    ("Silver", "tabela", "silver_vendas_itens_pedido", "granularidade reutilizável para análise"),
    ("Silver", "tabela", "silver_catalogo_produtos", "cadastro padronizado de produto"),
    ("Gold", "tabela", "gold_vendas_receita_mensal_produto_uf", "produto de dado para pergunta recorrente"),
    ("Gold", "tabela", "gold_vendas_kpis_mensais", "KPIs oficiais do mês")
], ["camada", "materializacao", "objeto", "promessa"])

show_title("3) Convenção de organização")
d(organizacao)


In [0]:
"""
PASSO: Evoluir a limpeza mínima da Live 1 para uma Silver reutilizável.
O QUE OBSERVAR:
  - reaproveitamos o mesmo pipeline-base da aula anterior
  - agora adicionamos uma regra explícita de deduplicação por chave
  - Silver não é mais só "dado arrumado"; é contrato técnico para o time trabalhar
VALIDAR:
  - tipos consistentes
  - status padronizado
  - duplicatas removidas com critério explícito
SINAL DE ERRO:
  - regra implícita ou limpeza manual fora do código

ASSUNÇÕES:
  - para preservar a sensação de evolução, mantemos a geografia da Live 1 em UF (e não região)
  - cada linha limpa representa um item de pedido
  - nesta continuação mantemos os 3 status da Live 1: pago, cancelado, pendente
"""

vendas_1 = standardize_column_names(vendas_bronze)

rename_map = {
    "qtd": "quantidade",
    "valor_total_r": "valor_total"
}

vendas_2 = vendas_1
for old, new in rename_map.items():
    if old in vendas_2.columns:
        vendas_2 = vendas_2.withColumnRenamed(old, new)

# Trim básico
v = vendas_2
for c, t in v.dtypes:
    if t == "string":
        v = v.withColumn(c, F.trim(F.col(c))) 

v = (
    v
    .withColumn("id_pedido", F.upper(F.col("id_pedido")))
    .withColumn("uf", F.upper(F.col("uf")))
    .withColumn("produto", F.lower(F.col("produto")))
    .withColumn("status", F.lower(F.col("status")))
)

v = v.withColumn(
    "status",
    F.when(F.col("status").rlike("cancel"), F.lit("cancelado"))
     .when(F.col("status").rlike("pend"), F.lit("pendente"))
     .otherwise(F.lit("pago"))
)

# Parse robusto de data: tenta vários formatos sem estourar erro
# Dessa vez, utilizando um padrão mais atual de resolução
# Observação: try_to_timestamp em PySpark aceita o formato como literal.
v = v.withColumn(
    "data_pedido",
    F.coalesce(
        F.try_to_timestamp("data_pedido", F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp("data_pedido", F.lit("dd/MM/yyyy HH:mm:ss")),
        F.try_to_timestamp(F.concat(F.col("data_pedido"), F.lit(":00")), F.lit("dd/MM/yyyy HH:mm:ss"))
    )
)

v = v.withColumn(
    "quantidade",
    F.when(
        (F.col("quantidade").isNull()) | (F.col("quantidade") == "") | (F.col("quantidade").rlike(r"^\s+$")),
        None
    ).otherwise(F.col("quantidade")).cast("int")
)

v = v.withColumn(
    "produto",
    F.when(
        (F.col("produto").isNull()) | (F.col("produto") == "") | (F.col("produto").rlike(r"^\s+$")),
        None
    ).otherwise(F.col("produto"))
)

valor_raw = F.col("valor_total")
valor_sem_ponto_milhar = F.when(valor_raw.contains(","), F.regexp_replace(valor_raw, "\\.", "")).otherwise(valor_raw)
valor_ponto_decimal = F.regexp_replace(valor_sem_ponto_milhar, ",", ".")
valor_sem_espacos = F.regexp_replace(valor_ponto_decimal, " ", "")

v = (
    v
    .withColumn("valor_total", valor_sem_espacos.cast(T.DecimalType(10, 2)))
    .withColumn("dt", F.to_date(F.col("data_pedido")))
    .withColumn("mes_referencia", F.trunc(F.col("data_pedido"), "MM"))
)

vendas_silver_pre_dedup = v.filter(
    F.col("produto").isNotNull() &
    F.col("quantidade").isNotNull() &
    F.col("data_pedido").isNotNull() &
    F.col("valor_total").isNotNull()
)

# Tudo feito até aqui foi feito para normalizar o conteúdo para deduplicação
# Uma tabela melhor preparada facilita sua vida
vendas_silver = vendas_silver_pre_dedup.dropDuplicates(["id_pedido", "produto", "data_pedido"]) #.drop_duplicates

show_title("4) Silver — continuação direta da limpeza mínima")
print("Linhas antes da deduplicação:", vendas_silver_pre_dedup.count())
print("Linhas depois da deduplicação:", vendas_silver.count())

duplicatas_antes = (
    vendas_silver_pre_dedup
    .groupBy("id_pedido", "produto", "data_pedido")
    .count()
    .filter(F.col("count") > 1)
)
print("Grupos duplicados antes da deduplicação:", duplicatas_antes.count())

print("Schema SILVER:")
vendas_silver.printSchema()

print("Amostra SILVER:")
d(vendas_silver.limit(20))

In [0]:
"""
PASSO: Organizar entidades Silver e gravar como tabelas.
* Lembre-se do modelo dimensional
O QUE OBSERVAR:
  - pedidos: nível de pedido
  - itens_pedido: granularidade reutilizável
  - produtos: cadastro padronizado
VALIDAR:
  - integridade básica entre itens e catálogo
  - checagens simples de completude
SINAL DE ERRO:
  - misturar agregado final com base técnica reutilizável
"""

produtos_1 = standardize_column_names(produtos_bronze)
produtos_silver = (
    produtos_1
    .select(
        F.lower(F.trim(F.col("produto"))).alias("produto"),
        F.initcap(F.trim(F.col("categoria"))).alias("categoria"),
        F.lower(F.trim(F.col("status_produto"))).alias("status_produto")
    )
    .dropDuplicates(["produto"])
)

itens_pedido_silver = vendas_silver.select(
    "id_pedido",
    "data_pedido",
    "dt",
    "mes_referencia",
    "uf",
    "produto",
    "quantidade",
    "valor_total",
    "status"
)

pedidos_silver = (
    itens_pedido_silver
    .groupBy("id_pedido")
    .agg(
        # Min e first para agrupar por pedido
        F.min("data_pedido").alias("data_pedido"),
        F.first("dt", ignorenulls=True).alias("dt"),
        F.first("mes_referencia", ignorenulls=True).alias("mes_referencia"),
        F.first("uf", ignorenulls=True).alias("uf"),
        F.first("status", ignorenulls=True).alias("status"),
        
        # Sumarizações
        F.sum("quantidade").alias("quantidade_total"),
        F.round(F.sum("valor_total"), 2).alias("valor_total_pedido"),
        F.count("produto").alias("qtd_itens_pedido")
    )
)

null_counts = vendas_silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in ["id_pedido", "data_pedido", "uf", "produto", "quantidade", "valor_total", "status"]
])

produtos_sem_cadastro = (
    itens_pedido_silver.select("produto").distinct()
    .join(produtos_silver.select("produto").distinct(), on="produto", how="left_anti")
)

show_title("5) Silver — entidades reutilizáveis")
print("Pedidos SILVER:")
d(pedidos_silver.limit(20))

print("Itens SILVER:")
d(itens_pedido_silver.limit(20))

print("Produtos SILVER:")
d(produtos_silver)

print("Nulos em colunas-chave da Silver:")
d(null_counts)

print("Produtos em itens sem cadastro no catálogo:")
d(produtos_sem_cadastro)

for tbl in [TBL_SILVER_PEDIDOS, TBL_SILVER_ITENS, TBL_SILVER_PRODUTOS, TBL_GOLD_RECEITA, TBL_GOLD_KPIS]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

pedidos_silver.write.mode("overwrite").format("delta").saveAsTable(TBL_SILVER_PEDIDOS)
itens_pedido_silver.write.mode("overwrite").format("delta").saveAsTable(TBL_SILVER_ITENS)
produtos_silver.write.mode("overwrite").format("delta").saveAsTable(TBL_SILVER_PRODUTOS)


In [0]:
"""
PASSO: Criar Gold para responder pergunta recorrente do negócio.
O QUE OBSERVAR:
  - Gold nasce da Silver, não do Bronze cru
  - a regra de negócio fica explícita e repetível
VALIDAR:
  - receita mensal por produto e UF
  - KPIs mensais de pedido válido
SINAL DE ERRO:
  - mudar a definição de métrica sem comunicar e sem versionar

ASSUNÇÃO:
  - pedido válido = status = 'pago'
  - cancelado e pendente não entram em receita/kpis gold
"""

base_gold = itens_pedido_silver.filter(F.col("status") == "pago")
# Expectativa do negócio:
# "Qual foi a receita mensal, quantidade de pedidos válidos e ticket médio por item para cada produto e UF?"
# O objetivo é fornecer uma visão consolidada da performance de vendas, permitindo identificar quais produtos tiveram maior receita em cada estado,
# quantos pedidos pagos foram realizados e qual o valor médio por item vendido, segmentado por mês, produto e UF.
gold_receita_mensal_produto_uf = (
    base_gold
    .groupBy("mes_referencia", "uf", "produto")
    .agg(
        F.round(F.sum("valor_total"), 2).alias("receita_total"),
        F.countDistinct("id_pedido").alias("qtd_pedidos_validos"),
        F.round(F.avg("valor_total"), 2).alias("ticket_medio_item")
    )
    .orderBy("mes_referencia", "uf", "produto")
)

# Expectativa do negócio:
# Qual foi a receita total, quantidade de pedidos válidos, quantidade de itens vendidos 
# e ticket médio por pedido em cada mês, considerando apenas pedidos pagos? 
# O objetivo é fornecer uma visão consolidada dos principais KPIs mensais das vendas, 
# permitindo acompanhar o desempenho financeiro, volume de vendas e valor médio dos pedidos 
# ao longo do tempo, com base em pedidos efetivamente concluídos
gold_kpis_mensais = (
    pedidos_silver
    .filter(F.col("status") == "pago")
    .groupBy("mes_referencia")
    .agg(
        F.round(F.sum("valor_total_pedido"), 2).alias("receita_total"),
        F.countDistinct("id_pedido").alias("qtd_pedidos_validos"),
        F.sum("quantidade_total").alias("qtd_itens"),
        F.round(F.avg("valor_total_pedido"), 2).alias("ticket_medio_pedido")
    )
    .orderBy("mes_referencia")
)

contrato_metricas = spark.createDataFrame([
    ("receita_total", "soma de valor_total apenas de pedidos com status pago"),
    ("qtd_pedidos_validos", "count distinct de id_pedido com status pago"),
    ("qtd_itens", "soma de quantidade_total de pedidos válidos"),
    ("ticket_medio_pedido", "média de valor_total_pedido para pedidos válidos"),
    ("ticket_medio_item", "média de valor_total na granularidade item")
], ["metrica", "definicao"])

gold_receita_mensal_produto_uf.write.mode("overwrite").format("delta").saveAsTable(TBL_GOLD_RECEITA)
gold_kpis_mensais.write.mode("overwrite").format("delta").saveAsTable(TBL_GOLD_KPIS)

show_title("6) Gold — produtos de dado")
print("Receita mensal por produto e UF:")
d(gold_receita_mensal_produto_uf.limit(30))

print("KPIs mensais:")
d(gold_kpis_mensais)

print("Contrato mínimo de métricas:")
d(contrato_metricas)


In [0]:
"""
PASSO: Comparar o papel de cada camada.
O QUE OBSERVAR:
  - Bronze guarda evidência e aceita imperfeição
  - Silver reduz ambiguidade e padroniza o trabalho
  - Gold responde pergunta recorrente com contrato explícito
VALIDAR:
  - encontrabilidade
  - reprodutibilidade
  - confiança
  (vamos subir aulas falando só sobre isso no Método PACO)
SINAL DE ERRO:
  - alguém novo no time ainda depender de reunião para descobrir o dado certo.
"""

resumo_camadas = spark.createDataFrame([
    ("Bronze", "arquivo", vendas_bronze.count(), "captura e evidência da entrada"),
    ("Silver", "tabela", spark.table(TBL_SILVER_ITENS).count(), "base limpa e reutilizável"),
    ("Gold", "tabela", spark.table(TBL_GOLD_RECEITA).count(), "produto de dado para consumo")
], ["camada", "materializacao", "qtd_linhas", "promessa"])

show_title("7) Antes vs depois — arquivo x tabela")
d(resumo_camadas)


In [0]:
%sql
SELECT
  formato,
  qtd_linhas,
  qtd_arquivos,
  tamanho_mb,
  tempo_leitura_seg
FROM bronze_benchmark
ORDER BY tempo_leitura_seg ASC, tamanho_mb ASC;


In [0]:
%sql
SELECT
  mes_referencia,
  uf,
  produto,
  receita_total,
  qtd_pedidos_validos,
  ticket_medio_item
FROM aulas_ao_vivo.live_20260309.gold_vendas_receita_mensal_produto_uf
ORDER BY mes_referencia, uf, receita_total DESC;


In [0]:
%sql
-- DROP SCHEMA aulas_ao_vivo.live_20260309 CASCADE